In [6]:
import cma
import numpy as np
from numbers import Real
import torch
from PIL import Image
import random
import requests
from io import BytesIO
import time
from matplotlib import cm
import matplotlib.pyplot as plt
from tqdm import tqdm
from os import path
import os
import json
from IPython.display import display, clear_output


# Utils imports
from utils.rasterize import *
from utils.load import get_segment_imgs, get_primitive_imgs
from utils.clipemb import CLIP_emb_from_IMG, CLIP_emb_from_TEXT
from utils.cma import clip_sol


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

WIDTH = 300
HEIGHT = 300

CONTENT_DIR = './data'
RESULTS_DIR = path.join(CONTENT_DIR, 'results')
SEGMENT_DIR = path.join(CONTENT_DIR, 'segment_img')
RW_PRIMITIVES_DIR = path.join(CONTENT_DIR, 'rw_primitives')
PRIMITIVE_DIR = path.join(CONTENT_DIR, 'primitives')

SEGMENT_IMGS = get_segment_imgs(SEGMENT_DIR, (WIDTH, HEIGHT))
RW_PRIMITIVES_IMGS = get_segment_imgs(RW_PRIMITIVES_DIR, (WIDTH, HEIGHT))
PRIMITIVE_IMGS = get_primitive_imgs(CONTENT_DIR, (WIDTH, HEIGHT))

In [7]:
prompt_text = "a cloudy night sky with a full moon"
prompt_embedding = CLIP_emb_from_TEXT(prompt_text)

# scan the folder and compute clip embeddings for each imaging, printing the embedding with the file name
for filename in os.listdir(path.join(CONTENT_DIR, 'final_imgs')):
    if filename.endswith('.png'):
        img_path = os.path.join(CONTENT_DIR, 'final_imgs', filename)
        img = Image.open(img_path).convert('RGB')
        img = img.resize((WIDTH, HEIGHT))
        clip_emb = CLIP_emb_from_IMG(img)
        print(f"File: {filename}, Cosine Similarity: {torch.nn.functional.cosine_similarity(clip_emb, prompt_embedding, dim=1)}")

File: CMA_NO_GAMMA_150.png, Cosine Similarity: tensor([0.3512])
File: PGPE_45.png, Cosine Similarity: tensor([0.2975])
File: GRAD_15.png, Cosine Similarity: tensor([0.3209])
File: GRAD_300.png, Cosine Similarity: tensor([0.2802])
File: CMA_NO_GAMMA_15.png, Cosine Similarity: tensor([0.3170])
File: PGPE_NO_GAMMA_285.png, Cosine Similarity: tensor([0.3317])
File: PGPE_NO_GAMMA_15.png, Cosine Similarity: tensor([0.2968])
File: CMA_150.png, Cosine Similarity: tensor([0.3089])
File: GRAD_NO_GAMMA_45.png, Cosine Similarity: tensor([0.3043])
File: CMA_15.png, Cosine Similarity: tensor([0.2592])
File: GRAD_NO_GAMMA_300.png, Cosine Similarity: tensor([0.3034])
File: PGPE_15.png, Cosine Similarity: tensor([0.2956])
File: PGPE_150.png, Cosine Similarity: tensor([0.3193])
File: PGPE_NO_GAMMA_45.png, Cosine Similarity: tensor([0.3104])
File: GRAD_NO_GAMMA_15.png, Cosine Similarity: tensor([0.3159])
File: CMA_45.png, Cosine Similarity: tensor([0.2514])
File: GRAD_45.png, Cosine Similarity: tensor([0